In [6]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from datetime import date
import re
from tqdm import tqdm
import time
from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr
from scipy.spatial.distance import jaccard, euclidean, cosine
import functools
import operator
import gzip

today = date.today()

**SNP**

**Transcription Factor**

**DHS**

**Chromatin State**

**Genes**

In [95]:
# 25kb bins
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)
print(bins_25kb)


# SNP data
snp_cols = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'pvalue', 'phenotype']
snp_df = pd.read_csv('../data/UKBB_all_traits_all_sig_associations_CT20220207.bed', delimiter='\t', names=snp_cols)
print(snp_df)

trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])
print(trait_codes_df)

# TF data

# DHS data
dhs_df = pd.read_pickle('../data/dhs_metadata.pkl')
print(dhs_df)

fn = gzip.GzipFile('../data/2018-06-08NC16_NNDSVD_Mixture.npy.gz', "r")
dhs_Mixture = np.load(fn)
dhs_Mixture_df = pd.DataFrame(dhs_Mixture, columns=dhs_df.index)
print(dhs_Mixture_df)

dhs_enrichment_df = pd.read_pickle('../data/dhs_enrichment_df_25kb.pkl')
print(dhs_enrichment_df)

# Chromatin State Data
epilogos_833_df = pd.read_pickle('../data/epilogos_state_data_18.pkl')
print(epilogos_833_df)

epilogos_states_df = pd.read_pickle('../data/epilogos_states_df')
print(epilogos_states_df)

epilogos_metadata = pd.read_csv('../data/All_833_biosamples/metadata.tsv', delimiter='\t')
print(epilogos_metadata)


# Gene Data
string_assoc = pd.read_pickle('../data/hg19-protein-associations.pkl')
print(string_assoc)

gene_hg19_bed = pd.read_pickle('../data/hg19_gene_bed.pkl')
print(gene_hg19_bed)

      seqname      start        end     ?
0        chr1       9000      34000  0.41
1        chr1      34000      59000  0.41
2        chr1      61000      86000  0.41
3        chr1      86000     111000  0.41
4        chr1     111000     136000  0.41
...       ...        ...        ...   ...
99991    chrX  155042800  155067800  0.41
99992    chrX  155067800  155092800  0.41
99993    chrX  155092800  155117800  0.41
99994    chrX  155117800  155142800  0.41
99995    chrX  155142800  155167800  0.41

[99996 rows x 4 columns]
         Chromosome         p1         p2 Reference SNP cluster ID  \
0             chr12  130300801  130300802               rs73150827   
1              chr4   99318162   99318163                rs1229984   
2              chr9    8303561    8303562              rs112120933   
3              chr1   46941503   46941504              rs141586057   
4             chr20   52932649   52932650              rs185961463   
...             ...        ...        ...         

         seqname      start        end   0   1   2   3   4   5   6  ...  823  \
0           chr1          0        200  18  18  18  18  18  18  18  ...   18   
1           chr1        200        400  18  18  18  18  18  18  18  ...   18   
2           chr1        400        600  18  18  18  18  18  18  18  ...   18   
3           chr1        600        800  18  18  18  18  18  18  18  ...   18   
4           chr1        800       1000  18  18  18  18  18  18  18  ...   18   
...          ...        ...        ...  ..  ..  ..  ..  ..  ..  ..  ...  ...   
15181503    chrX  155269400  155269600  18  18  18  18  18  18  18  ...   18   
15181504    chrX  155269600  155269800  18  18  18  18  18  18  18  ...   18   
15181505    chrX  155269800  155270000  18  18  18  18  18  18  18  ...   18   
15181506    chrX  155270000  155270200  18  18  18  18  18  18  18  ...   18   
15181507    chrX  155270200  155270400  18  18  18  18  18  18  18  ...   18   

          824  825  826  827  828  829 

# 25kb 

In [43]:
# GATA1 25kb regions
gene = 'GATA1'

gene_chr = gene_hg19_bed.loc[gene][0]
gene_start = gene_hg19_bed.loc[gene][1]
gene_end = gene_hg19_bed.loc[gene][2]

print(gene_hg19_bed.loc[gene])
print(gene_start)
print(gene_end)
# print(bins_25kb[bins_25kb['seqname']==gene_chr].loc[96548])

print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1])
print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0])

region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0]

0                            chrX
1                        48644961
2                        48652715
4                             200
5                               +
6                 ENST00000376670
7                            4170
8                  protein-coding
9          GATA binding protein 1
10                       48649516
11                       48652571
12        92,239,378,146,126,516,
13    0,4536,5289,5768,6617,7238,
Name: GATA1, dtype: object
48644961
48652715
seqname        chrX
start      48583800
end        48608800
?           7.49548
Name: 96547, dtype: object
seqname        chrX
start      48647800
end        48672800
?           7.49905
Name: 96548, dtype: object


In [72]:
region_index = region.name
region_chr = region.seqname
region_start = region.start
region_end = region.end

In [67]:
### SNP ###

snp_chr = snp_df.loc[snp_df['Chromosome'] == region_chr]
snp_in_region = snp_chr[(snp_chr['p1'] >= region_start) & (snp_chr['p2'] <= region_end)]

trait_codes_temp = trait_codes_df.set_index(trait_codes_df['phenotype'])

snp_in_region['description'] = snp_in_region['phenotype'].map(trait_codes_temp['description'])
snp_in_region

/tmp/ipykernel_136593/782567659.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snp_in_region['description'] = snp_in_region['phenotype'].map(trait_codes_temp['description'])


,Chromosome,p1,p2,Reference SNP cluster ID,pvalue,phenotype,description
1364253,chrX,48653200,48653201,rs782651911,1.252070e-09,20003_1140872112,Treatment/medication code: epanutin 25mg capsule
1494527,chrX,48650300,48650301,rs186413167,3.778510e-12,20003_1140882728,Treatment/medication code: otomize ear spray
1739839,chrX,48670508,48670509,rs185602812,6.217330e-15,20003_1141168108,Treatment/medication code: isotard 25xl m/r ta...
2223162,chrX,48651557,48651558,rs191287036,5.724230e-10,22601_11313221,"Job coding: financial services manager, charte..."
2223163,chrX,48668251,48668252,rs782024726,5.033350e-10,22601_11313221,"Job coding: financial services manager, charte..."
2283659,chrX,48649536,48649537,rs782740386,2.328120e-08,22601_21232866,"Job coding: electrical engineer, generating en..."
2374008,chrX,48669796,48669797,rs6609777,9.248590e-10,22601_32143084,"Job coding: medical radiographer, x-ray operat..."
2442517,chrX,48651557,48651558,rs191287036,7.724450e-11,22601_35623207,Job coding: personnel or industrial relations ...
2442518,chrX,48668251,48668252,rs782024726,1.037720e-10,22601_35623207,Job coding: personnel or industrial relations ...
2518516,chrX,48669796,48669797,rs6609777,9.759760e-09,22601_52222928,"Job coding: tool maker, tool fitter, jig and t..."


In [73]:
### DHS ###
# Which DHSs are in the region
dhs_metadata_chr = dhs_df[dhs_df['seqname'] == region_chr]
dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 
print(dhs_metadata_region)

# Component Enrichments in the region
print(dhs_enrichment_df.loc[region_index])

           seqname     start       end  mean_signal  numsamples    summit  \
identifier                                                                  
X.38053       chrX  48648398  48648560     0.963202         483  48648470   
X.38054       chrX  48648420  48648740     1.211198          25  48648590   
X.38055       chrX  48648717  48648767     0.086107           1  48648750   
X.38056       chrX  48648860  48648955     0.213544           1  48648910   
X.38058       chrX  48648870  48649100     0.137287           1  48648990   
X.38059       chrX  48649060  48649227     0.183267           1  48649170   
X.38061       chrX  48649203  48649398     0.675666          52  48649310   
X.38062       chrX  48652025  48652220     0.330941          16  48652110   
X.38064       chrX  48652254  48652400     0.287811           5  48652310   
X.38065       chrX  48654300  48655100     0.365473           2  48654630   
X.380662      chrX  48655160  48655480     0.477974           1  48655310   

In [85]:
### EPILOGOS ###
epilogos_chr = epilogos_states_df[epilogos_states_df['seqname'] == region_chr]
epilogos_in_region = epilogos_chr[(epilogos_chr['start'] >= region_start) & (epilogos_chr['end'] <= region_end)]

epilogos_region = epilogos_in_region.loc[:,list(range(1,19))].sum(axis=0)
epilogos_region

1     59.435964
2      5.986387
3     18.471882
4      1.830059
5      3.040723
6     85.517700
7      0.121624
8      0.037042
9      3.310406
10     1.650296
11     3.363812
12    -0.383685
13    -1.656135
14    -0.009520
15     0.000004
16    -0.167308
17     3.158272
18   -20.257399
dtype: float64

In [89]:
epilogos_metadata

,zero_index,one_index,short_name,long_name,hex,rgba,color
0,0,1,TssA,Active TSS,#ff0000,"rgba(255,0,0,1)",Red
1,1,2,TssFlnk,Flanking TSS,#ff4500,"rgba(255,69,0,1)",Orange Red
2,2,3,TssFlnkU,Flanking TSS Upstream,#ff4500,"rgba(255,69,0,1)",Orange Red
3,3,4,TssFlnkD,Flanking TSS Downstream,#ff4500,"rgba(255,69,0,1)",Orange Red
4,4,5,Tx,Strong transcription,#008000,"rgba(0,128,0,1)",Green
5,5,6,TxWk,Weak transcription,#006400,"rgba(0,100,0,1)",DarkGreen
6,6,7,EnhG1,Genic Enhancer 1,#c2e105,"rgba(194,225,5,1)",GreenYellow
7,7,8,EnhG2,Genic Enhancer 2,#c2e105,"rgba(194,225,5,1)",GreenYellow
8,8,9,EnhA1,Active Enhancer 1,#ffc34d,"rgba(255,195,77,1)",Orange
9,9,10,EnhA2,Active Enhancer 2,#ffc34d,"rgba(255,195,77,1)",Orange


In [117]:
### GENE ###
gene_chr = gene_hg19_bed.loc[gene_hg19_bed['chr'] == region_chr]
genes_in_region = gene_chr[((gene_chr['start'] >= region_start) & (gene_chr['start'] <= region_end)) | ((gene_chr['end'] <= region_end) & (gene_chr['end'] >= region_start))]
print(genes_in_region)

             chr     start       end    4 strand                6      7  \
gene_names                                                                 
HDAC6       chrX  48660401  48683386  212      +  ENST00000334136  14064   
GATA1       chrX  48644961  48652715  200      +  ENST00000376670   4170   

                         8              descriptor        10        11  \
gene_names                                                               
HDAC6       protein-coding   histone deacetylase 6  48661099  48683022   
GATA1       protein-coding  GATA binding protein 1  48649516  48652571   

                                                           12  \
gene_names                                                      
HDAC6       148,123,129,89,85,41,97,98,105,69,127,66,60,90...   
GATA1                                 92,239,378,146,126,516,   

                                                           13  
gene_names                                                     
HDAC6    

In [119]:
hdac6_start = gene_hg19_bed.loc['HDAC6'].start
hdac6_end = gene_hg19_bed.loc['HDAC6'].end
dhs_metadata_chr = dhs_df[dhs_df['seqname'] == 'chrX']
dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 
print(dhs_metadata_region.component.sort_values(ascending=False))

identifier
X.380685    Vascular / endothelial
X.38053           Tissue invariant
X.380682                 Stromal B
X.380678            Renal / cancer
X.380691            Renal / cancer
X.38065      Primitive / embryonic
X.380662     Primitive / embryonic
X.38054      Primitive / embryonic
X.38059                     Neural
X.380675                    Neural
X.380695       Myeloid / erythroid
X.380688       Myeloid / erythroid
X.38064        Myeloid / erythroid
X.380668       Myeloid / erythroid
X.38056            Musculoskeletal
X.380698                  Lymphoid
X.380671                  Lymphoid
X.38062                   Lymphoid
X.38061                  Digestive
X.38058                  Digestive
X.38055                  Digestive
X.380665                 Digestive
Name: component, dtype: object
